# AMD Screening Case Study

This notebook presents the two case studies accompanying the AMD screening feature paper.
It demonstrates how Average Minimum Distance (AMD) and Pointwise Distance Distribution
(PDD/EMD) metrics can be used to detect duplicate and similar crystal structures across
and within databases.

**Case Study 1 — Cross-database deduplication**  
Compare structures retrieved from the Materials Project (MP) and the Crystallography Open
Database (COD) to identify entries that describe the same or very similar crystal.

**Case Study 2 — Within-database similarity screening**  
Screen OQMD structures to discover groups of identical, very similar, and merely similar
entries and rank candidates by geometric diversity.

---
**Reproducibility note (Online Methods)**  
All CIF files used in these case studies are stored in the repository under:
```
tests/fixtures/cif_files/mp/     (12 structures)
tests/fixtures/cif_files/cod/    (20 structures)
tests/fixtures/cif_files/oqmd/   (39 structures)
```
No external downloads or API keys are required to reproduce the results.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

# Locate workspace root automatically
workspace_root = Path.cwd().resolve()
while workspace_root != workspace_root.parent:
    if (workspace_root / 'Features').is_dir() and (workspace_root / 'Information_Units').is_dir():
        break
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from Information_Units.Predictors.AMD.AMDPredictor import AMDPredictor

# Fixture directories (relative to workspace root)
CIF_MP   = workspace_root / 'tests' / 'fixtures' / 'cif_files' / 'mp'
CIF_COD  = workspace_root / 'tests' / 'fixtures' / 'cif_files' / 'cod'
CIF_OQMD = workspace_root / 'tests' / 'fixtures' / 'cif_files' / 'oqmd'


class NotebookLogger:
    def log(self, message, level='info'):
        tag = level.upper().ljust(7)
        print(f'[{tag}] {message}')


logger = NotebookLogger()
print(f'Workspace root : {workspace_root}')
print(f'MP  CIF files  : {len(list(CIF_MP.glob("*.cif")))}')
print(f'COD CIF files  : {len(list(CIF_COD.glob("*.cif")))}')
print(f'OQMD CIF files : {len(list(CIF_OQMD.glob("*.cif")))}')

## Shared helpers

In [ ]:
def load_cif_files(directory: Path):
    """Return a list of (filename, cif_path) for all .cif files in a directory."""
    return sorted(
        [(p.stem, p) for p in directory.glob('*.cif')],
        key=lambda x: x[0],
    )


def run_amd(cif_paths, k=100, metric='chebyshev', logger=None):
    """Run AMDPredictor on a list of Path objects; return result dict."""
    predictor = AMDPredictor(predictor_name='amd_notebook', k=k, metric=metric, logger=logger)
    result = predictor.predict({'input_data': [str(p) for p in cif_paths]})
    return result


def pairwise_to_df(result, name_map=None):
    """
    Convert AMD result dict to a tidy DataFrame of pairwise distances.

    Parameters
    ----------
    result : dict
        Output from AMDPredictor.predict().
    name_map : dict, optional
        Mapping from file path string to display label.

    Returns
    -------
    pd.DataFrame
    """
    first = (result.get('results') or [{}])[0]
    if first.get('status') != 'success':
        raise RuntimeError(f'AMD prediction failed: {first.get("error")}')

    rows = []
    for pair in first['properties']['pairwise_distances']:
        if 'error' in pair:
            continue
        path1 = pair['crystal_1_file']
        path2 = pair['crystal_2_file']
        label1 = name_map.get(path1, Path(path1).stem) if name_map else Path(path1).stem
        label2 = name_map.get(path2, Path(path2).stem) if name_map else Path(path2).stem
        rows.append({
            'structure_1': label1,
            'structure_2': label2,
            'pdd_emd': pair['pdd_emd_distance'],
            'amd_dist': pair['amd_distance'],
            'identical':    pair.get('identical', False),
            'very_similar': pair.get('very_similar', False),
            'similar':      pair.get('similar', False),
        })
    return pd.DataFrame(rows)


def build_distance_matrix(df_pairs, structures):
    """
    Convert tidy pair DataFrame to a square distance matrix (PDD/EMD).

    Returns a (matrix, labels) tuple where `matrix[i, j]` is the PDD/EMD
    distance between structures[i] and structures[j].
    """
    n = len(structures)
    idx = {name: i for i, name in enumerate(structures)}
    mat = np.full((n, n), np.nan)
    np.fill_diagonal(mat, 0.0)

    for _, row in df_pairs.iterrows():
        i, j = idx.get(row['structure_1']), idx.get(row['structure_2'])
        if i is not None and j is not None:
            mat[i, j] = row['pdd_emd']
            mat[j, i] = row['pdd_emd']
    return mat


def plot_distance_matrix(matrix, labels, title='Pairwise PDD/EMD Distance Matrix',
                         vmax=None, figsize=None, ax=None):
    """Plot a symmetric distance matrix as an annotated heatmap."""
    n = len(labels)
    if figsize is None:
        side = max(5, n * 0.55)
        figsize = (side, side)

    if vmax is None:
        finite = matrix[~np.isnan(matrix) & (matrix > 0)]
        vmax = float(np.percentile(finite, 95)) if finite.size else 1.0

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(matrix, cmap='viridis_r', aspect='auto', vmin=0, vmax=vmax)
    if standalone:
        plt.colorbar(im, ax=ax, label='PDD/EMD distance')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(title, fontsize=11)

    if n <= 20:
        for i in range(n):
            for j in range(n):
                v = matrix[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                            fontsize=6, color='white' if v > vmax * 0.5 else 'black')

    if standalone:
        plt.tight_layout()
        plt.show()


def print_summary(title, df_pairs):
    """Print a human-readable summary of pairwise AMD screening results."""
    print(f'\n=== {title} ===')
    print(f'Total pairs compared   : {len(df_pairs)}')
    print(f'Identical    (PDD < 1e-6) : {df_pairs["identical"].sum()}')
    print(f'Very similar (PDD < 0.10) : {(df_pairs["very_similar"] & ~df_pairs["identical"]).sum()}')
    print(f'Similar      (PDD < 0.50) : {(df_pairs["similar"] & ~df_pairs["very_similar"]).sum()}')
    print(f'Distinct     (PDD >= 0.50): {(~df_pairs["similar"]).sum()}')
    print()
    close = df_pairs[df_pairs['pdd_emd'] < 0.5].sort_values('pdd_emd')
    if not close.empty:
        print('Closest pairs (PDD/EMD < 0.5):')
        display(close[['structure_1', 'structure_2', 'pdd_emd', 'amd_dist',
                        'identical', 'very_similar']].head(15))

---
## Case Study 1 — Cross-database Deduplication: MP vs COD

Both the Materials Project and the COD archive crystal structures from the experimental and
computational literature. The same compound is frequently deposited in both databases under
different identifiers, in different settings, or with slight numerical differences.

This case study shows that AMD screening reliably identifies these duplicates without
requiring element-type information, formula matching, or space-group comparison.

**Setup**: 12 MP structures + 20 COD structures = 32 total, 496 pairwise comparisons.

In [ ]:
mp_entries  = load_cif_files(CIF_MP)
cod_entries = load_cif_files(CIF_COD)

all_cs1_entries = mp_entries + cod_entries
all_cs1_paths   = [p for _, p in all_cs1_entries]
cs1_name_map    = {str(p): name for name, p in all_cs1_entries}
cs1_labels      = [name for name, _ in all_cs1_entries]

print(f'MP structures  : {len(mp_entries)}')
print(f'COD structures : {len(cod_entries)}')
print(f'Total          : {len(all_cs1_entries)}')
print(f'Expected pairs : {len(all_cs1_entries) * (len(all_cs1_entries) - 1) // 2}')

In [ ]:
print('Running AMD screening on Case Study 1 (MP vs COD)...')
cs1_result = run_amd(all_cs1_paths, k=100, metric='chebyshev', logger=logger)
cs1_df = pairwise_to_df(cs1_result, name_map=cs1_name_map)

print(f'\nSuccessfully computed {len(cs1_df)} pairwise distances.')
cs1_df.head()

In [ ]:
print_summary('Case Study 1 — Cross-database Deduplication (MP vs COD)', cs1_df)

In [ ]:
cs1_matrix = build_distance_matrix(cs1_df, cs1_labels)
plot_distance_matrix(
    cs1_matrix, cs1_labels,
    title='Case Study 1: MP vs COD — PDD/EMD Distance Matrix',
    figsize=(14, 12),
)

### Cross-database pairs only

The diagonal blocks (MP–MP and COD–COD) may contain near-duplicates within the same
database. The off-diagonal block shows which MP entries are geometrically equivalent to COD
entries — these are the true cross-database duplicates.

In [ ]:
mp_names  = {name for name, _ in mp_entries}
cod_names = {name for name, _ in cod_entries}

cross_df = cs1_df[
    (cs1_df['structure_1'].isin(mp_names) & cs1_df['structure_2'].isin(cod_names)) |
    (cs1_df['structure_1'].isin(cod_names) & cs1_df['structure_2'].isin(mp_names))
].copy()

cross_close = cross_df[cross_df['pdd_emd'] < 0.5].sort_values('pdd_emd')

print(f'Cross-database pairs total  : {len(cross_df)}')
print(f'Cross-database identical    : {cross_df["identical"].sum()}')
print(f'Cross-database very similar : {(cross_df["very_similar"] & ~cross_df["identical"]).sum()}')
print(f'Cross-database similar      : {(cross_df["similar"] & ~cross_df["very_similar"]).sum()}')

if not cross_close.empty:
    print('\nClosest cross-database pairs (PDD/EMD < 0.5):')
    display(cross_close[['structure_1', 'structure_2', 'pdd_emd', 'amd_dist', 'identical', 'very_similar']].head(15))

---
## Case Study 2 — Within-database Similarity Screening: OQMD

Large databases such as OQMD contain many structurally related entries — sometimes the
same compound in different settings, ionic variants of a prototype, or slightly distorted
polymorphs. Identifying these groups before a downstream workflow avoids redundant
calculations and biased statistics.

This case study uses AMD screening to partition 39 OQMD structures into groups of
identical, very similar, and similar entries, and ranks candidates by geometric novelty
(i.e. maximum minimum distance to any other candidate).

**Setup**: 39 OQMD structures → 741 pairwise comparisons.

In [ ]:
oqmd_entries = load_cif_files(CIF_OQMD)
oqmd_paths   = [p for _, p in oqmd_entries]
oqmd_name_map = {str(p): name for name, p in oqmd_entries}
oqmd_labels  = [name for name, _ in oqmd_entries]

print(f'OQMD structures : {len(oqmd_entries)}')
print(f'Expected pairs  : {len(oqmd_entries) * (len(oqmd_entries) - 1) // 2}')

In [ ]:
print('Running AMD screening on Case Study 2 (OQMD within-database)...')
cs2_result = run_amd(oqmd_paths, k=100, metric='chebyshev', logger=logger)
cs2_df = pairwise_to_df(cs2_result, name_map=oqmd_name_map)

print(f'\nSuccessfully computed {len(cs2_df)} pairwise distances.')
cs2_df.head()

In [ ]:
print_summary('Case Study 2 — Within-database Screening (OQMD)', cs2_df)

In [ ]:
cs2_matrix = build_distance_matrix(cs2_df, oqmd_labels)
plot_distance_matrix(
    cs2_matrix, oqmd_labels,
    title='Case Study 2: OQMD — PDD/EMD Distance Matrix',
    figsize=(16, 14),
)

### Diversity ranking

A structure is considered *geometrically diverse* if its minimum PDD/EMD distance to any
other structure in the set is large. The ranking below can be used to select a maximally
diverse subset, e.g. as the seed set for an active-learning campaign.

In [ ]:
# For each structure, compute its minimum distance to all other structures
min_dists = {}
for label in oqmd_labels:
    partner_distances = cs2_df[
        (cs2_df['structure_1'] == label) | (cs2_df['structure_2'] == label)
    ]['pdd_emd'].values
    # Exclude self-comparisons (distance = 0)
    non_zero = partner_distances[partner_distances > 1e-9]
    min_dists[label] = float(np.min(non_zero)) if non_zero.size > 0 else 0.0

diversity_df = pd.DataFrame([
    {'structure': k, 'min_pdd_emd_to_others': v}
    for k, v in min_dists.items()
]).sort_values('min_pdd_emd_to_others', ascending=False).reset_index(drop=True)

print('Diversity ranking (most geometrically distinct structures first):')
display(diversity_df.head(20))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(diversity_df)), diversity_df['min_pdd_emd_to_others'], color='steelblue')
ax.set_xticks(range(len(diversity_df)))
ax.set_xticklabels(diversity_df['structure'], rotation=90, fontsize=7)
ax.set_xlabel('Structure')
ax.set_ylabel('Min PDD/EMD distance to others')
ax.set_title('OQMD Structure Diversity (higher = more unique)')
plt.tight_layout()
plt.show()

### Similarity clusters

Structures with PDD/EMD < 0.1 are grouped into clusters. Each cluster represents a set of
very similar (or identical) entries that could be treated as a single representative in
downstream computations.

In [ ]:
from collections import defaultdict

SIMILARITY_THRESHOLD = 0.1  # PDD/EMD threshold for "very similar"

# Build adjacency using union-find
parent = {label: label for label in oqmd_labels}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

for _, row in cs2_df[cs2_df['pdd_emd'] < SIMILARITY_THRESHOLD].iterrows():
    union(row['structure_1'], row['structure_2'])

clusters = defaultdict(list)
for label in oqmd_labels:
    clusters[find(label)].append(label)

cluster_list = sorted(clusters.values(), key=len, reverse=True)

print(f'Similarity threshold (PDD/EMD) : {SIMILARITY_THRESHOLD}')
print(f'Total structures               : {len(oqmd_labels)}')
print(f'Number of clusters             : {len(cluster_list)}')
print(f'Singleton clusters             : {sum(1 for c in cluster_list if len(c) == 1)}')
print(f'Multi-member clusters          : {sum(1 for c in cluster_list if len(c) > 1)}')
print()
print('Multi-member clusters (very similar groups):')
for i, cluster in enumerate(c for c in cluster_list if len(c) > 1):
    print(f'  Cluster {i + 1} ({len(cluster)} members): {cluster}')

---
## Combined: AMD vs PDD/EMD Metric Comparison

The AMD metric (Chebyshev distance between AMD vectors) and the PDD/EMD metric measure
complementary aspects of structural similarity. This section plots both metrics against
each other to highlight where they agree and disagree.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in [
    (axes[0], cs1_df, 'Case Study 1 (MP vs COD)'),
    (axes[1], cs2_df, 'Case Study 2 (OQMD)'),
]:
    colors = df['pdd_emd'].apply(
        lambda x: 'red' if x < 1e-6 else ('orange' if x < 0.1 else ('gold' if x < 0.5 else 'steelblue'))
    )
    ax.scatter(df['pdd_emd'], df['amd_dist'], c=colors, alpha=0.6, s=20, edgecolors='none')
    ax.set_xlabel('PDD/EMD distance')
    ax.set_ylabel('AMD distance (Chebyshev)')
    ax.set_title(title)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red',      label='Identical (PDD < 1e-6)'),
        Patch(facecolor='orange',   label='Very similar (PDD < 0.1)'),
        Patch(facecolor='gold',     label='Similar (PDD < 0.5)'),
        Patch(facecolor='steelblue',label='Distinct'),
    ]
    ax.legend(handles=legend_elements, fontsize=8)

plt.suptitle('AMD distance vs PDD/EMD distance', fontsize=12)
plt.tight_layout()
plt.show()

---
## Save results

In [ ]:
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)

cs1_df.to_csv(output_dir / 'case_study_1_mp_vs_cod_pairwise.csv', index=False)
cs2_df.to_csv(output_dir / 'case_study_2_oqmd_pairwise.csv', index=False)
diversity_df.to_csv(output_dir / 'case_study_2_oqmd_diversity_ranking.csv', index=False)

with open(output_dir / 'case_study_1_raw_result.json', 'w') as fh:
    json.dump(cs1_result, fh, indent=2)

with open(output_dir / 'case_study_2_raw_result.json', 'w') as fh:
    json.dump(cs2_result, fh, indent=2)

print('Results saved to:', output_dir.resolve())